
# Nevergrad first call
Quick notebook to exercise the optimization step (Nevergrad) with the cached pipeline. Adjust overrides to control runtime and target.


In [ ]:

# Ensure repo on sys.path (works when run from CLI or inside the notebook directory)
import sys
from pathlib import Path

# Start from CWD; if __file__ defined (exported), use that
repo_root = Path.cwd()
if '__file__' in globals():
    repo_root = Path(__file__).resolve().parents[2]

# Climb parents until we find the top-level nevermore/configs marker
marker = Path('nevermore/configs/default.yaml')
for parent in [repo_root] + list(repo_root.parents):
    if (parent / marker).exists():
        repo_root = parent
        break

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print('Repo root1:', repo_root)


In [ ]:

import json
import pandas as pd
from nevermore.pipeline import NevermorePipeline


STAGE_ORDER = [
    "ingest",
    "features",
    "optimization",
    "retrieval",
    "visualization",
    "docking",
    "admet",
    "report",
]

# Config and quick overrides
config_path = repo_root / 'nevermore' / 'configs' / 'default.yaml'
override_budget = 400  # set None to use config value (default 300)
override_sample_index = None  # e.g., 10
override_target_sequence = None  # supply a sequence string to override dataset entry
override_baseline_smiles = None  # supply a SMILES string to override dataset entry

pipe = NevermorePipeline(config_path=config_path)
if override_budget is not None:
    pipe.config.optimization.budget = override_budget
if override_sample_index is not None:
    pipe.config.optimization.sample_index = override_sample_index
if override_target_sequence is not None:
    pipe.config.optimization.target_sequence = override_target_sequence
if override_baseline_smiles is not None:
    pipe.config.optimization.baseline_smiles = override_baseline_smiles

results = pipe.run(up_to='report', verbose=True)
opt_res = results.get('report')
print('Optimization signature:', opt_res.signature)
print('Outputs:', opt_res.outputs)
print('Details:', json.dumps(opt_res.details, indent=2))


In [ ]:

# Inspect optimization summary CSV and target manifest
from pathlib import Path
summary_path = Path(opt_res.outputs.get('summary', ''))
manifest_path = Path(opt_res.outputs.get('manifest', ''))
if summary_path.exists():
    display(pd.read_csv(summary_path).head())
if manifest_path.exists():
    print(manifest_path.read_text())
